In [1]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [2]:
train_df= pd.read_csv("samsum-train.csv")
val_df= pd.read_csv("samsum-validation.csv")

In [3]:
train_df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_df["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_df.shape

(14732, 3)

In [6]:
val_df.shape

(818, 3)

## Random Sampling

In [7]:
train_df= train_df.sample(n= 4000, random_state= 42,).reset_index(drop= True)
val_df= val_df.sample(n= 500, random_state= 42,).reset_index(drop= True)

In [8]:
train_df.shape

(4000, 3)

## Preprocessing

In [9]:
import re
def clean_text(text):
    text = re.sub(r'\r\n', ' ', text)  # Replace newlines with space
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text= re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text= text.strip()  # Remove leading and trailing whitespace
    return text


In [10]:
train_df['dialogue']= train_df['dialogue'].apply(clean_text)
train_df['summary']= train_df['summary'].apply(clean_text)

val_df['dialogue']= val_df['dialogue'].apply(clean_text)
val_df['summary']= val_df['summary'].apply(clean_text)


In [11]:
train_df["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:  Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

## Tokenization

In [12]:
tokenizer= T5Tokenizer.from_pretrained("t5-small")


## Raw data => Tokenized inputs for fine tuning

In [13]:
def tokenize(data):
    inputs= tokenizer(data['dialogue'], padding= 'max_length', truncation= True, max_length= 512)
    targets= tokenizer(data['summary'], padding= 'max_length', truncation= True, max_length= 128)
    inputs['labels']= targets['input_ids']
    return inputs

In [14]:
train_df_set= train_df.apply(tokenize, axis= 1).tolist()
val_df_set= val_df.apply(tokenize, axis= 1).tolist()

In [15]:
train_df_set[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## Working with Model

In [16]:
model= T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [17]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
Device name: NVIDIA GeForce RTX 3060 Laptop GPU


##  Training

In [19]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="epoch",
    warmup_steps=500,
    fp16=True
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_df_set,
    eval_dataset=val_df_set,
)

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.442152,0.456112
2,0.476239,0.427800
3,0.447770,0.419905
4,0.433170,0.416516
5,0.424613,0.414430
6,0.420510,0.414308


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=1.1074090321858725, metrics={'train_runtime': 980.1398, 'train_samples_per_second': 24.486, 'train_steps_per_second': 3.061, 'total_flos': 3248203235328000.0, 'train_loss': 1.1074090321858725, 'epoch': 6.0})